# Adapting mixing matrices

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/19-matrix-scaling.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

## Goal of this chapter

Many countries have no empirically collected contact-survey data, so we often
have to infer contact patterns in one setting from another. There are many
ways to do this. Contact between population groups probably depends on many
things — the age structure and make-up of workplaces and schools, and a wide
range of social and cultural factors — which are outside the scope of this
series. Here we adjust for one thing only: the difference in age structure
between the setting where the data were collected and the setting we want to
model.

## Example to consider

We adapt the United Kingdom POLYMOD matrix to the recent age structure of
India. In one sense this is a poor example: the two settings differ hugely in
ways other than their population distribution. But their population
distributions also differ a great deal, which is what makes the example useful.

The matrix is the POLYMOD Great Britain matrix from Mossong *et al.* (2008),
supporting table
[doi:10.1371/journal.pmed.0050074.st005](https://doi.org/10.1371/journal.pmed.0050074.st005)
(Creative Commons Attribution licence), as transcribed in the source textbook.
summer4 ships no survey data: this page embeds the extract it uses, so nothing
is downloaded when the documentation builds.

India's population by age is the one the source textbook uses.

## Collating our data

First, the age structure of each setting. The differences are considerable.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from summer4.epi import ContactAdaptation, ContactMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

# POLYMOD Great Britain, as published: rows are the AGE OF CONTACT and columns
# the AGE GROUP OF PARTICIPANT.
PUBLISHED = np.array(
    [
        [1.92, 0.65, 0.41, 0.24, 0.46, 0.73, 0.67, 0.83, 0.24, 0.22, 0.36, 0.20, 0.20, 0.26, 0.13],
        [0.95, 6.64, 1.09, 0.73, 0.61, 0.75, 0.95, 1.39, 0.90, 0.16, 0.30, 0.22, 0.50, 0.48, 0.20],
        [0.48, 1.31, 6.85, 1.52, 0.27, 0.31, 0.48, 0.76, 1.00, 0.69, 0.32, 0.44, 0.27, 0.41, 0.33],
        [0.33, 0.34, 1.03, 6.71, 1.58, 0.73, 0.42, 0.56, 0.85, 1.16, 0.70, 0.30, 0.20, 0.48, 0.63],
        [0.45, 0.30, 0.22, 0.93, 2.59, 1.49, 0.75, 0.63, 0.77, 0.87, 0.88, 0.61, 0.53, 0.37, 0.33],
        [0.79, 0.66, 0.44, 0.74, 1.29, 1.83, 0.97, 0.71, 0.74, 0.85, 0.88, 0.87, 0.67, 0.74, 0.33],
        [0.97, 1.07, 0.62, 0.50, 0.88, 1.19, 1.67, 0.89, 1.02, 0.91, 0.92, 0.61, 0.76, 0.63, 0.27],
        [1.02, 0.98, 1.26, 1.09, 0.76, 0.95, 1.53, 1.50, 1.32, 1.09, 0.83, 0.69, 1.02, 0.96, 0.20],
        [0.55, 1.00, 1.14, 0.94, 0.73, 0.88, 0.82, 1.23, 1.35, 1.27, 0.89, 0.67, 0.94, 0.81, 0.80],
        [0.29, 0.54, 0.57, 0.77, 0.97, 0.93, 0.57, 0.80, 1.32, 1.87, 0.61, 0.80, 0.61, 0.59, 0.57],
        [0.33, 0.38, 0.40, 0.41, 0.44, 0.85, 0.60, 0.61, 0.71, 0.95, 0.74, 1.06, 0.59, 0.56, 0.57],
        [0.31, 0.21, 0.25, 0.33, 0.39, 0.53, 0.68, 0.53, 0.55, 0.51, 0.82, 1.17, 0.85, 0.85, 0.33],
        [0.26, 0.25, 0.19, 0.24, 0.19, 0.34, 0.40, 0.39, 0.47, 0.55, 0.41, 0.78, 0.65, 0.85, 0.57],
        [0.09, 0.11, 0.12, 0.20, 0.19, 0.22, 0.13, 0.30, 0.23, 0.13, 0.21, 0.28, 0.36, 0.70, 0.60],
        [0.14, 0.15, 0.21, 0.10, 0.24, 0.17, 0.15, 0.41, 0.50, 0.71, 0.53, 0.76, 0.47, 0.74, 1.47],
    ]
)
AGE_GROUPS = list(range(0, 75, 5))  # 0-4, 5-9, ..., 65-69, 70+
BANDS = (*AGE_GROUPS, math.inf)

UK_2006 = np.array(
    [3458060, 3556024, 3824317, 3960916, 3911291, 3762213, 4174675, 4695853,
     4653082, 3986098, 3620216, 3892985, 3124676, 2706365, 6961183],
    dtype=float,
)

INDIA = np.array(
    [116879506, 117982127, 126155952, 126045565, 122504805, 117397268, 112176099,
     103460177, 90219895, 79440281, 68875962, 59256267, 48890529, 38260284, 52459668],
    dtype=float,
)
uk_age_props = pd.Series(UK_2006 / UK_2006.sum(), index=AGE_GROUPS)
india_age_props = pd.Series(INDIA / INDIA.sum(), index=AGE_GROUPS)
pd.DataFrame({"india": india_age_props, "uk": uk_age_props}).plot(
    title="Age structure of the two settings",
    labels={"index": "age", "value": "proportion of population"},
)

## Get the base matrix

Next, the base matrix for the United Kingdom that we want to adapt. It carries
the UK population it was surveyed in as its `source_population`.

In [ ]:
uk_matrix = ContactMatrix.from_array(PUBLISHED.T, BANDS, source_population=UK_2006)
pd.DataFrame(uk_matrix.values, index=AGE_GROUPS, columns=AGE_GROUPS).plot(
    kind="imshow",
    title="United Kingdom (POLYMOD)",
    labels={"x": "contact age", "y": "respondent age", "color": "contacts"},
)

## Adaptation

For each age band, take the ratio of its share of India's population to its
share of the UK's. Then multiply each **column** of the matrix — the age of the
person contacted — by that band's ratio: people meet more of whoever is more
common. That is the same as multiplying the matrix on the right by a diagonal
matrix of the ratios.

{meth}`~summer4.epi.ContactMatrix.adapt` does this with `kind="density"`: the
rate at which two particular people meet is kept, and contacts with each band
scale with its share of the new population. The cell checks it against the
matrix product written out by hand.

In [ ]:
india_uk_ratios = india_age_props / uk_age_props
adjusted_matrix = uk_matrix.adapt(INDIA, kind=ContactAdaptation.DENSITY)
np.testing.assert_allclose(
    adjusted_matrix.values, uk_matrix.values @ np.diag(india_uk_ratios.to_numpy())
)
india_uk_ratios.round(2).to_frame("India share / UK share").T

## Results

Compare this matrix with the unadjusted one from {doc}`17-empiric-surveys`.
Contacts are now much more skewed toward the younger age groups, as we would
expect for a population as young as India's.

In [ ]:
pd.DataFrame(adjusted_matrix.values, index=AGE_GROUPS, columns=AGE_GROUPS).plot(
    kind="imshow",
    title="Adapted to India's age structure",
    labels={"x": "contact age", "y": "respondent age", "color": "contacts"},
)

In [ ]:
fig = go.Figure(data=[go.Surface(x=AGE_GROUPS, y=AGE_GROUPS, z=adjusted_matrix.values)])
fig.update_layout(
    title="Adapted matrix as a surface",
    width=800,
    height=600,
    scene=dict(xaxis_title="contact age", yaxis_title="respondent age", zaxis_title="contacts"),
)
fig

This matrix is probably not ideal for modelling an infection in India. But
using the UK matrix as reported, with India's population distribution, would be
plain wrong.

## A second convention (summer4)

*This section is not in the source textbook.* The adjustment above keeps the
chance that two particular people meet, so the number of contacts each person
reports changes with the population. The literature also uses the opposite
convention: keep each band's *number* of contacts per person, and let only
whom they are with shift. `adapt(..., kind="frequency")` — summer4's default —
does that, rescaling each row of the density result back to its original total.
The two conventions disagree whenever the populations differ, so choose one
deliberately. The chart compares contacts per person under each.

In [ ]:
frequency_matrix = uk_matrix.adapt(INDIA, kind=ContactAdaptation.FREQUENCY)
per_person = pd.DataFrame(
    {
        "UK survey": uk_matrix.total(),
        "India, density (above)": adjusted_matrix.total(),
        "India, frequency": frequency_matrix.total(),
    },
    index=pd.Index(AGE_GROUPS, name="age"),
)
np.testing.assert_allclose(per_person["India, frequency"], per_person["UK survey"])
per_person.plot(
    title="Contacts per person under the two conventions",
    labels={"index": "respondent age", "value": "contacts per day", "variable": ""},
    markers=True,
)